In [15]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

print("Loading data...")
df_sample = pd.read_csv('HI-Small_Trans.csv', nrows=100000)
print(f"Data loaded successfully! Shape: {df_sample.shape}")

# Separating the labels 
y_true = df_sample['Is Laundering'] 

# Dropping unused columns (hardcoded)
X = df_sample.drop(columns=['Is Laundering', 'Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1'])

#Encoding text columns into numbers
encoder = LabelEncoder()

for col in ['Receiving Currency', 'Payment Currency', 'Payment Format']:
    if col in X.columns:
        X[col] = encoder.fit_transform(X[col])

# first check of what it looks like
print("\nHere is the clean, numeric data the model will see:")
print(X.head())

print("Training Isolation Forest")
# Contamination is our estimated fraud rate (0.1%)
iso_forest = IsolationForest(n_estimators=100, contamination=0.001, random_state=42)

#Train the model (no labels!)
iso_forest.fit(X)

# Isolation Forest outputs 1 for normal, and -1 for anomalies (fraud)
predictions = iso_forest.predict(X)

# Let's map those predictions to 0 (normal) and 1 (fraud) so it matches our y_true labels
y_pred = np.where(predictions == -1, 1, 0)

# Calculate Minority-Class F1 (pos_label=1 means we only care about catching the 1s/fraud)
f1 = f1_score(y_true, y_pred, pos_label=1)
precision = precision_score(y_true, y_pred, pos_label=1)
recall = recall_score(y_true, y_pred, pos_label=1)

print("\n--- RESULTS ON 100k SUBSAMPLE ---")
print(f"Minority-Class F1 Score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

# Print the full report to see how bad the Accuracy Trap is!
print("\nFull Classification Report:")
print(classification_report(y_true, y_pred))

Loading data...
Data loaded successfully! Shape: (100000, 11)

Here is the clean, numeric data the model will see:
   Amount Received  Receiving Currency  Amount Paid  Payment Currency  \
0          3697.34                   9      3697.34                 9   
1             0.01                   9         0.01                 9   
2         14675.57                   9     14675.57                 9   
3          2806.97                   9      2806.97                 9   
4         36682.97                   9     36682.97                 9   

   Payment Format  
0               5  
1               3  
2               5  
3               5  
4               5  
Training Isolation Forest

--- RESULTS ON 100k SUBSAMPLE ---
Minority-Class F1 Score: 0.0000
Precision: 0.0000
Recall: 0.0000

Full Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     99995
           1       0.00      0.00      0.00         5

    acc

In [16]:
# Load the first 100,000 rows
print("Loading data")
df_sample = pd.read_csv('HI-Small_Trans.csv', nrows=1000000)

# Let's count how many criminals are actually in this new sample
print("\nFraud count in this sample:")
print(df_sample['Is Laundering'].value_counts())
print(f"Data loaded successfully! Shape: {df_sample.shape}")

#Separate the Labels
y_true = df_sample['Is Laundering'] 

# Dropping columns the model shouldn't use
X = df_sample.drop(columns=['Is Laundering', 'Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1'])

# Encoding text columns into numbers
encoder = LabelEncoder()

#checking if the column exists before encoding to avoid errors
for col in ['Receiving Currency', 'Payment Currency', 'Payment Format']:
    if col in X.columns:
        X[col] = encoder.fit_transform(X[col])

print("\nHere is the clean, numeric data the model will see:")
print(X.head())

#Initializing the Unsupervised Model
print("Training Isolation Forest")
#Contamination is our estimated fraud rate (0.1%)
iso_forest = IsolationForest(n_estimators=100, contamination=0.001, random_state=42)

# Training the model 
iso_forest.fit(X)

# Isolation Forest outputs 1 for normal, and -1 for anomalies (fraud)
predictions = iso_forest.predict(X)

# Let's map those predictions to 0 (normal) and 1 (fraud) so it matches our y_true labels
y_pred = np.where(predictions == -1, 1, 0)

# Calculate Minority-Class F1 (pos_label=1 means we only care about catching the 1s/fraud)
f1 = f1_score(y_true, y_pred, pos_label=1)
precision = precision_score(y_true, y_pred, pos_label=1)
recall = recall_score(y_true, y_pred, pos_label=1)

print("\n--- RESULTS ON 100k SUBSAMPLE ---")
print(f"Minority-Class F1 Score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

# Print the full report just to see how bad the Accuracy Trap is!
print("\nFull Classification Report:")
print(classification_report(y_true, y_pred))

Loading data

Fraud count in this sample:
Is Laundering
0    999425
1       575
Name: count, dtype: int64
Data loaded successfully! Shape: (1000000, 11)

Here is the clean, numeric data the model will see:
   Amount Received  Receiving Currency  Amount Paid  Payment Currency  \
0          3697.34                  12      3697.34                12   
1             0.01                  12         0.01                12   
2         14675.57                  12     14675.57                12   
3          2806.97                  12      2806.97                12   
4         36682.97                  12     36682.97                12   

   Payment Format  
0               5  
1               3  
2               5  
3               5  
4               5  
Training Isolation Forest

--- RESULTS ON 100k SUBSAMPLE ---
Minority-Class F1 Score: 0.0025
Precision: 0.0020
Recall: 0.0035

Full Classification Report:
              precision    recall  f1-score   support

           0       1.00  

In [17]:
# Testing the three different net sizes: 0.1%, 0.5%, and 1%
contamination_rates = [0.001, 0.005, 0.01]

print("Starting Tuning for Isolation Forest...\n")

for rate in contamination_rates:
    print(f"--- Testing Contamination Rate: {rate * 100}% ---")
    
    # Initializing the model with the new rate
    # We also increase n_estimators to 200 (more trees = more stable predictions)
    iso_forest = IsolationForest(n_estimators=200, contamination=rate, random_state=42)
    
    #Training the model
    iso_forest.fit(X)
    
    predictions = iso_forest.predict(X)
    y_pred = np.where(predictions == -1, 1, 0)
    
    # Grading
    f1 = f1_score(y_true, y_pred, pos_label=1)
    precision = precision_score(y_true, y_pred, pos_label=1)
    recall = recall_score(y_true, y_pred, pos_label=1)
    
    print(f"F1 Score:  {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    
    # Calculate exactly how many criminals were caught
    criminals_caught = int(recall * 575) #we know there are 575 total
    print(f"Criminals Caught: {criminals_caught} out of 575\n")

print("Tuning Complete!")

Starting Tuning for Isolation Forest...

--- Testing Contamination Rate: 0.1% ---


F1 Score:  0.0026
Precision: 0.0020
Recall:    0.0035
Criminals Caught: 2 out of 575

--- Testing Contamination Rate: 0.5% ---
F1 Score:  0.0011
Precision: 0.0006
Recall:    0.0052
Criminals Caught: 3 out of 575

--- Testing Contamination Rate: 1.0% ---
F1 Score:  0.0008
Precision: 0.0004
Recall:    0.0070
Criminals Caught: 4 out of 575

Tuning Complete!


In [18]:
# Master Results DataFrame
# We take the original sample and add the model's best predictions to it
results_df = df_sample.copy()

# iso_forest is currently trained on best contamination rate of 0.001)
best_predictions = iso_forest.predict(X)
results_df['Model_Guess'] = np.where(best_predictions == -1, 1, 0)

# Checking where the model guessed right (True Positives)
# A True Positive is when Is Laundering == 1 AND Model_Guess == 1
results_df['Caught_By_Model'] = ((results_df['Is Laundering'] == 1) & (results_df['Model_Guess'] == 1))

# empty column for the specific Typology (Fan-Out, Cycle, etc.)
# We will fill this using the HI-Small_Patterns.txt file later
results_df['Laundering_Typology'] = "Normal" 

print("Results Table Prepared: Here are the 4 criminals we caught:")
# Let's look at just the rows where the model successfully caught a criminal
caught_criminals = results_df[results_df['Caught_By_Model'] == True]
print(caught_criminals[['Amount Received', 'Receiving Currency', 'Is Laundering', 'Model_Guess', 'Laundering_Typology']])

Results Table Prepared: Here are the 4 criminals we caught:
        Amount Received Receiving Currency  Is Laundering  Model_Guess  \
216900     3.708012e+07                Yen              1            1   
401034     4.087365e+08               Euro              1            1   
557728     8.485314e+10              Ruble              1            1   
825119     6.641449e+10              Rupee              1            1   

       Laundering_Typology  
216900              Normal  
401034              Normal  
557728              Normal  
825119              Normal  


In [19]:
# Let's peek at the labels
print("--- Peeking inside HI-Small_Patterns.txt ---")
try:
    with open('HI-Small_Patterns.txt', 'r') as file:
        for i in range(10):
            print(file.readline().strip())
except FileNotFoundError:
    print("Error: Could not find 'HI-Small_Patterns.txt'.")

--- Peeking inside HI-Small_Patterns.txt ---
BEGIN LAUNDERING ATTEMPT - FAN-OUT:  Max 16-degree Fan-Out
2022/09/01 00:06,021174,800737690,012,80011F990,2848.96,Euro,2848.96,Euro,ACH,1
2022/09/01 04:33,021174,800737690,020,80020C5B0,8630.40,Euro,8630.40,Euro,ACH,1
2022/09/01 09:14,021174,800737690,020,80006A5E0,35642.49,Yuan,35642.49,Yuan,ACH,1
2022/09/01 09:56,021174,800737690,00220,8007A5B70,5738987.96,US Dollar,5738987.96,US Dollar,ACH,1
2022/09/01 11:28,021174,800737690,001244,80093C0D0,7254.53,US Dollar,7254.53,US Dollar,ACH,1
2022/09/01 13:13,021174,800737690,00513,80078E200,6990.87,US Dollar,6990.87,US Dollar,ACH,1
2022/09/01 14:11,021174,800737690,020,80066B990,12536.92,Euro,12536.92,Euro,ACH,1
2022/09/02 15:40,021174,800737690,00410,8002CC310,3511.82,Euro,3511.82,Euro,ACH,1
2022/09/02 21:23,021174,800737690,01292,8004030A0,16135.09,US Dollar,16135.09,US Dollar,ACH,1


In [20]:
print("1. Parsing the Typologies from HI-Small_Patterns.txt")

# reading the text file and building a dictionary mapping
pattern_data = []
current_typology = "Unknown"

with open('HI-Small_Patterns.txt', 'r') as file:
    for line in file:
        line = line.strip()
        
        # If the line is a header, update our "current typology" state
        if line.startswith("BEGIN LAUNDERING ATTEMPT -"):
            # This extracts just the word "FAN-OUT", "CYCLE", etc.
            current_typology = line.split('-')[1].split(':')[0].strip()
            
        # If the line starts with a year (2022), it is a transaction record
        elif line.startswith("2022"):
            fields = line.split(',')
            if len(fields) >= 11:
                # Grabbing identifiers for this transaction
                pattern_data.append({
                    'Timestamp': fields[0],
                    'Account': fields[2],      # Sender Account
                    'Account.1': fields[4],    # Receiver Account
                    'Mapped_Typology': current_typology
                })

# Convert our extracted labels into a DataFrame
patterns_df = pd.DataFrame(pattern_data)

print("2. Merging Typologies onto the Results Table")
# We do a 'Left Join' to attach the typologies to the 1-million row sample based on the 3 keys
results_df = results_df.merge(
    patterns_df[['Timestamp', 'Account', 'Account.1', 'Mapped_Typology']], 
    on=['Timestamp', 'Account', 'Account.1'], 
    how='left'
)

# Replacing the empty 'Normal' column we made earlier with our new mapped data
results_df['Laundering_Typology'] = results_df['Mapped_Typology'].fillna("Normal")
results_df = results_df.drop(columns=['Mapped_Typology'])

# Let's see what crimes are actually in our 1-million row sample!
print("\n--- CRIME BREAKDOWN IN 1 MILLION ROWS ---")
print(results_df['Laundering_Typology'].value_counts())

1. Parsing the Typologies from HI-Small_Patterns.txt
2. Merging Typologies onto the Results Table

--- CRIME BREAKDOWN IN 1 MILLION ROWS ---
Laundering_Typology
Normal       999584
FAN              85
GATHER           83
STACK            77
CYCLE            59
BIPARTITE        57
SCATTER          38
RANDOM           17
Name: count, dtype: int64


In [21]:
# testing the two most important typologies:
typologies_to_test = ['FAN', 'CYCLE']

print("--- ISOLATED TYPOLOGY SCORES (TABULAR BASELINE) ---")

for crime in typologies_to_test:
    # Isolate the environment
    # We filter the data to ONLY include Innocent transactions + THIS specific crime
    subset = results_df[(results_df['Is Laundering'] == 0) | (results_df['Laundering_Typology'] == crime)]
    
    y_true_sub = subset['Is Laundering']
    y_pred_sub = subset['Model_Guess']
    
    # grading the test strictly on this environment
    f1 = f1_score(y_true_sub, y_pred_sub, pos_label=1, zero_division=0)
    precision = precision_score(y_true_sub, y_pred_sub, pos_label=1, zero_division=0)
    recall = recall_score(y_true_sub, y_pred_sub, pos_label=1, zero_division=0)
    
    print(f"\n Typology: {crime} ")
    print(f"Criminals in this test: {len(subset[subset['Is Laundering'] == 1])}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")

--- ISOLATED TYPOLOGY SCORES (TABULAR BASELINE) ---

 Typology: FAN 
Criminals in this test: 84
F1 Score:  0.0000
Precision: 0.0000
Recall:    0.0000

 Typology: CYCLE 
Criminals in this test: 44
F1 Score:  0.0000
Precision: 0.0000
Recall:    0.0000


In [22]:
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import KMeans

print("--- TRAINING MODEL 2: LOCAL OUTLIER FACTOR (LOF) ---")
print(" (LOF is calculating distances for 1 million rows. )")
# We use the same contamination rate (0.1%) to keep the fight fair
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.001)
# LOF uses fit_predict() instead of fit() then predict()
lof_predictions = lof.fit_predict(X)
results_df['LOF_Guess'] = np.where(lof_predictions == -1, 1, 0)
print(" LOF Training Complete!\n")


print("--- TRAINING MODEL 3: K-MEANS (Distance Threshold) ---")
# Grouping the data into 10 normal behavior clusters
kmeans = KMeans(n_clusters=10, random_state=42, n_init='auto')
kmeans.fit(X)

# Calculating the distance from every transaction to its cluster center
distances = np.min(kmeans.transform(X), axis=1)

# Imposing the Distance Threshold
# We find the distance threshold that strictly separates the furthest 0.1% of points
threshold = np.percentile(distances, 99.9)

# 4. If distance > threshold, it's an anomaly (1). Otherwise, normal (0).
results_df['KMeans_Guess'] = np.where(distances > threshold, 1, 0)
print(" K-Means Training Complete!\n")


print("======================================================")
print(" FINAL TABULAR  RESULTS (1 MILLION ROWS) ")
print("======================================================")

# 'Model_Guess' is our Isolation Forest from earlier
models_to_test = {
    'Isolation Forest': 'Model_Guess', 
    'Local Outlier Factor': 'LOF_Guess', 
    'K-Means': 'KMeans_Guess'
}
typologies = ['FAN', 'CYCLE']

for model_name, column_name in models_to_test.items():
    print(f"\n Evaluating: {model_name}")
    
    # 1. Global Score (Aggregated)
    global_f1 = f1_score(results_df['Is Laundering'], results_df[column_name], pos_label=1, zero_division=0)
    print(f"  Overall Global F1: {global_f1:.4f}")
    
    # 2. Isolated Typology Scores
    for crime in typologies:
        subset = results_df[(results_df['Is Laundering'] == 0) | (results_df['Laundering_Typology'] == crime)]
        y_true_sub = subset['Is Laundering']
        y_pred_sub = subset[column_name]
        
        f1 = f1_score(y_true_sub, y_pred_sub, pos_label=1, zero_division=0)
        print(f"  Strict {crime} F1:    {f1:.4f}")


--- TRAINING MODEL 2: LOCAL OUTLIER FACTOR (LOF) ---
 (LOF is calculating distances for 1 million rows. )


c:\Users\yboulal\AppData\Local\miniconda3\envs\aifoundry\Lib\site-packages\sklearn\neighbors\_lof.py:325: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(


 LOF Training Complete!

--- TRAINING MODEL 3: K-MEANS (Distance Threshold) ---
 K-Means Training Complete!

 FINAL TABULAR  RESULTS (1 MILLION ROWS) 

 Evaluating: Isolation Forest
  Overall Global F1: 0.0008
  Strict FAN F1:    0.0000
  Strict CYCLE F1:    0.0000

 Evaluating: Local Outlier Factor
  Overall Global F1: 0.0000
  Strict FAN F1:    0.0000
  Strict CYCLE F1:    0.0000

 Evaluating: K-Means
  Overall Global F1: 0.0025
  Strict FAN F1:    0.0000
  Strict CYCLE F1:    0.0000
